# Subscription LTV, Cohorts and Break-Even CAC
### KKBox Churn Prediction Challenge (WSDM Cup 2018)

**Data note:** This notebook uses the synthetic CSV files committed with this repository: `members_v3.csv`, `transactions_v2.csv`, and `train_v2.csv`. These are demonstration data, not actual KKBox customer records. The real competition dataset is not included.

**Goal:** build cohort retention curves, project LTV by acquisition channel (`registered_via`), and derive modelled break-even CAC ceilings. No actual marketing-spend data is included, so the analysis does not measure observed channel CAC or LTV:CAC performance.


## 1. Setup & Data Load

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

pd.set_option("display.width", 120)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

CHANNEL_NAMES = {3: "Organic/Web", 4: "App Store", 7: "Paid Social", 9: "Partner Bundle", 13: "Affiliate/Promo"}
CHANNEL_COLORS = {"Organic/Web": "#2563eb", "App Store": "#16a34a", "Paid Social": "#dc2626",
                   "Partner Bundle": "#9333ea", "Affiliate/Promo": "#ea580c"}

members = pd.read_csv("members_v3.csv")
txn = pd.read_csv("transactions_v2.csv")
train = pd.read_csv("train_v2.csv")

print(f"members_v3.csv: {len(members):,} rows")
print(f"transactions_v2.csv: {len(txn):,} rows")
print(f"train_v2.csv: {len(train):,} rows  |  observed churn rate: {train['is_churn'].mean():.1%}")


In [ ]:

# Parse dates, map channel codes to readable labels
members["registration_init_time"] = pd.to_datetime(members["registration_init_time"], format="%Y%m%d")
txn["transaction_date"] = pd.to_datetime(txn["transaction_date"], format="%Y%m%d")
txn["membership_expire_date"] = pd.to_datetime(txn["membership_expire_date"], format="%Y%m%d")
members["channel"] = members["registered_via"].map(CHANNEL_NAMES).fillna(members["registered_via"].astype(str))

DATA_CUTOFF = txn["transaction_date"].max()
print(f"Transaction data cutoff: {DATA_CUTOFF.date()}")

members[["msno", "city", "registered_via", "channel", "registration_init_time"]].head()


## 2. Building the Revenue Ledger

Merge transactions onto member records so every payment carries its acquisition channel and
registration date. From that we derive **cohort age** — months elapsed since registration at the
time of each transaction — which is the x-axis for every curve in this notebook.

In [ ]:

ledger = txn.merge(members[["msno", "registration_init_time", "channel", "city"]], on="msno", how="left")
ledger = ledger.dropna(subset=["registration_init_time"])

ledger["age_start_month"] = ((ledger["transaction_date"] - ledger["registration_init_time"]).dt.days // 30).clip(lower=0)

# Cap the "active until" date at the data cutoff before computing age_end_month.
# Without this, a long prepaid plan (e.g. 410 days) bought shortly before the cutoff has a
# membership_expire_date far in the FUTURE relative to the cutoff, which would mark the user
# "active" in age-buckets that haven't chronologically occurred yet by the cutoff - inconsistent
# with how eligibility is defined below, and it produces a spurious retention spike at the
# corresponding age (all channels showing >=100% retention at the same age).
ledger["_active_until"] = ledger["membership_expire_date"].clip(upper=DATA_CUTOFF)
ledger["age_end_month"] = ((ledger["_active_until"] - ledger["registration_init_time"]).dt.days // 30).clip(lower=0)

cohort_size = members.groupby("channel")["msno"].nunique().rename("cohort_size").sort_values(ascending=False)
print("Users acquired per channel:")
cohort_size


## 3. Cohort Retention Curves

For a valid cohort curve, the denominator at each age must be restricted to users who've actually
had *time* to reach that age by the data cutoff — otherwise recently-acquired users get counted as
"still eligible" for ages they haven't lived through yet, and the curve is biased. We compute an
**eligible cohort size** per (channel, age) pair to correct for this right-censoring.

In [ ]:

MAX_AGE_MONTHS = 30
MIN_ELIGIBLE = 400  # below this sample size we stop trusting the observed curve and extrapolate instead

elig_rows = []
for channel, grp in members.groupby("channel"):
    for age in range(MAX_AGE_MONTHS):
        threshold = DATA_CUTOFF - pd.Timedelta(days=30 * age)
        n_eligible = (grp["registration_init_time"] <= threshold).sum()
        elig_rows.append((channel, age, n_eligible))
eligible = pd.DataFrame(elig_rows, columns=["channel", "age_month", "eligible_users"])

# Active-user expansion: each transaction covers the age-bucket range [start, end),
# exclusive of the end bucket -- inclusive would double-count the boundary month
# for every ~30-day renewal, which silently inflates retention almost everywhere.
rows = []
for row in ledger.itertuples(index=False):
    start, end = int(row.age_start_month), int(row.age_end_month)
    end = max(end, start + 1)
    for m in range(start, end):
        rows.append((row.channel, m, row.msno))
active = pd.DataFrame(rows, columns=["channel", "age_month", "msno"]).drop_duplicates()

retention = active.groupby(["channel", "age_month"])["msno"].nunique().reset_index(name="active_users")
retention = retention.merge(eligible, on=["channel", "age_month"])
retention["retention_rate"] = (retention["active_users"] / retention["eligible_users"]).clip(upper=1.0)
retention.head(8)


In [ ]:

fig, ax = plt.subplots(figsize=(10, 6))
for channel in cohort_size.index:
    r = retention[retention.channel == channel].sort_values("age_month")
    r_solid = r[r.eligible_users >= MIN_ELIGIBLE]
    r_dashed = r[r.eligible_users < MIN_ELIGIBLE]
    ax.plot(r_solid.age_month, r_solid.retention_rate * 100, color=CHANNEL_COLORS[channel], label=channel, linewidth=2)
    if len(r_dashed):
        ax.plot(r_dashed.age_month, r_dashed.retention_rate * 100, color=CHANNEL_COLORS[channel],
                linewidth=1.2, linestyle=":", alpha=0.6)

ax.set_xlabel("Months since registration")
ax.set_ylabel("% of eligible cohort still active")
ax.set_title("Retention Curves by Acquisition Channel\n(dotted = low-sample tail, <400 eligible users)")
ax.set_ylim(0, 105)
ax.legend(loc="lower left", frameon=False)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
plt.tight_layout()
plt.show()


## 4. Cohort Heatmap (Book of Business)

A classic SaaS cohort heatmap: each row is a registration month, each column is months-since-signup,
and the color is retention. Reads diagonally — you can see whether *newer* cohorts retain better or
worse than older ones, independent of channel.

In [ ]:

ledger["reg_month"] = ledger["registration_init_time"].dt.to_period("M").astype(str)

rows = []
for row in ledger.itertuples(index=False):
    start, end = int(row.age_start_month), int(row.age_end_month)
    end = max(end, start + 1)
    for m in range(start, end):
        rows.append((row.reg_month, m, row.msno))
active_cal = pd.DataFrame(rows, columns=["reg_month", "age_month", "msno"]).drop_duplicates()
cohort_counts = active_cal.groupby(["reg_month", "age_month"])["msno"].nunique().reset_index(name="n")

cohort_totals = members.groupby(members["registration_init_time"].dt.to_period("M").astype(str))["msno"].nunique()
cohort_counts["cohort_total"] = cohort_counts["reg_month"].map(cohort_totals)
cohort_counts["retention"] = cohort_counts["n"] / cohort_counts["cohort_total"]

heat = cohort_counts.pivot(index="reg_month", columns="age_month", values="retention")
heat = heat.loc[sorted(heat.index)]

fig, ax = plt.subplots(figsize=(13, 7))
im = ax.imshow(heat.values * 100, aspect="auto", cmap="YlGnBu", vmin=0, vmax=100)
ax.set_xticks(range(0, heat.shape[1], 2))
ax.set_xticklabels(heat.columns[::2])
ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels(heat.index, fontsize=7)
ax.set_xlabel("Months since registration")
ax.set_ylabel("Registration cohort (month)")
ax.set_title("Retention Heatmap: Registration Cohort x Age (all channels)")
cbar = fig.colorbar(im, ax=ax, label="% retained")
plt.tight_layout()
plt.show()


## 5. LTV Curves by Channel

**Key methodological point:** the LTV numerator must be restricted to the same eligible population
as the denominator at every age. Naively summing all revenue collected at each age bucket (which
mixes contributions from cohorts of very different maturity) and dividing by a shrinking eligible
count would badly inflate the tail of the curve. Instead, for each user we cap their observable
history at *their own* cohort age (registration to data cutoff), then average cumulative revenue
only across users who've actually lived long enough to reach each age.

In [ ]:

members["max_obs_age"] = ((DATA_CUTOFF - members["registration_init_time"]).dt.days // 30).clip(lower=0, upper=MAX_AGE_MONTHS - 1)

user_age_revenue = ledger.groupby(["msno", "age_start_month"])["actual_amount_paid"].sum().reset_index()
user_age_revenue = user_age_revenue.rename(columns={"age_start_month": "age_month"})

pivot = user_age_revenue.pivot_table(index="msno", columns="age_month", values="actual_amount_paid", fill_value=0)
pivot = pivot.reindex(columns=range(MAX_AGE_MONTHS), fill_value=0)
cum_pivot = pivot.cumsum(axis=1)

max_obs = members.set_index("msno")["max_obs_age"]
channel_map = members.set_index("msno")["channel"]

ltv_rows = []
for channel in cohort_size.index:
    ch_users = channel_map[channel_map == channel].index
    ch_cum = cum_pivot.loc[cum_pivot.index.intersection(ch_users)]
    ch_max_obs = max_obs.loc[ch_cum.index]
    for m in range(MAX_AGE_MONTHS):
        mask = ch_max_obs >= m
        n_elig = int(mask.sum())
        if n_elig == 0:
            continue
        avg_ltv = ch_cum.loc[mask.index[mask], m].mean()
        ltv_rows.append((channel, m, n_elig, avg_ltv))

ltv_df = pd.DataFrame(ltv_rows, columns=["channel", "age_month", "eligible_users", "ltv_observed"])

# incremental (non-cumulative) revenue per active user per period, for extrapolation
incr_revenue = ledger.groupby(["channel", "age_start_month"])["actual_amount_paid"].sum().reset_index()
incr_revenue = incr_revenue.rename(columns={"age_start_month": "age_month", "actual_amount_paid": "incremental_revenue"})
ltv_df = ltv_df.merge(incr_revenue, on=["channel", "age_month"], how="left")
ltv_df = ltv_df.merge(retention[["channel", "age_month", "active_users"]], on=["channel", "age_month"])
ltv_df["arpu_per_active_user"] = ltv_df["incremental_revenue"] / ltv_df["active_users"]

ltv_df[ltv_df.channel == "Organic/Web"].head(10)


## 6. Extrapolating LTV Beyond the Observed Window

Cohorts that registered recently haven't had time to age into month 18, 24, etc., so the observed
curve gets noisy and eventually runs out past the reliable sample threshold. To project a full
12/24-month LTV for every channel, we:

1. Fit an **exponential decay** to the reliable portion of the retention curve (log-linear regression
   → monthly survival multiplier).
2. Take the **steady-state ARPU per active user per period** from the reliable window.
3. Project forward: `retention(t) = retention(t_last) × decay^(t - t_last)`, and add
   `retention(t) × steady_ARPU` to cumulative LTV for each future period.

In [ ]:

PROJECT_TO = 24
extrap = {}

for channel in cohort_size.index:
    r = retention[retention.channel == channel].sort_values("age_month")
    r_reliable = r[(r.eligible_users >= MIN_ELIGIBLE) & (r.age_month >= 1)]

    x = r_reliable["age_month"].values
    y = np.log(r_reliable["retention_rate"].clip(lower=1e-4).values)
    b, a = np.polyfit(x, y, 1)
    monthly_decay = np.exp(b)

    last_reliable_age = int(r_reliable["age_month"].max())
    last_reliable_retention = float(r[r.age_month == last_reliable_age]["retention_rate"].iloc[0])

    arpu_window = ltv_df[(ltv_df.channel == channel) & (ltv_df.age_month.between(1, last_reliable_age))]
    steady_arpu = arpu_window["arpu_per_active_user"].mean()

    ltv_at_last_reliable = float(ltv_df[(ltv_df.channel == channel) & (ltv_df.age_month == last_reliable_age)]["ltv_observed"].iloc[0])

    curve = {age: float(ltv_df[(ltv_df.channel == channel) & (ltv_df.age_month == age)]["ltv_observed"].iloc[0])
             for age in ltv_df[ltv_df.channel == channel]["age_month"] if age <= last_reliable_age}

    retention_t, cum_ltv = last_reliable_retention, ltv_at_last_reliable
    for t in range(last_reliable_age + 1, PROJECT_TO + 1):
        retention_t *= monthly_decay
        cum_ltv += retention_t * steady_arpu
        curve[t] = cum_ltv

    extrap[channel] = dict(monthly_decay=monthly_decay, last_reliable_age=last_reliable_age,
                            steady_arpu=steady_arpu, curve=curve)

summary = pd.DataFrame({
    ch: dict(monthly_retention_decay=v["monthly_decay"], last_reliable_age_mo=v["last_reliable_age"],
             steady_arpu_ntd=round(v["steady_arpu"], 1))
    for ch, v in extrap.items()
}).T
summary


In [ ]:

fig, ax = plt.subplots(figsize=(10, 6))
for channel in cohort_size.index:
    curve = extrap[channel]["curve"]
    last_reliable = extrap[channel]["last_reliable_age"]
    ages = sorted(curve.keys())
    solid_ages = [a for a in ages if a <= last_reliable]
    dashed_ages = [a for a in ages if a >= last_reliable]
    ax.plot(solid_ages, [curve[a] for a in solid_ages], color=CHANNEL_COLORS[channel], linewidth=2, label=channel)
    ax.plot(dashed_ages, [curve[a] for a in dashed_ages], color=CHANNEL_COLORS[channel], linewidth=2, linestyle="--")

ax.set_xlabel("Months since registration")
ax.set_ylabel("Cumulative LTV per user (NTD)")
ax.set_title("LTV Curves by Channel\n(solid = observed, dashed = extrapolated)")
ax.legend(loc="upper left", frameon=False)
plt.tight_layout()
plt.show()


## 7. Break-Even CAC by Channel

Break-even CAC at a horizon *h* is simply `LTV(h)` — the most you could spend to acquire a user in
that channel and still break even by month *h*. Most subscription businesses target a healthy
margin, not just breakeven, so we also show the **max affordable CAC at a 3:1 LTV:CAC target**
(a common SaaS benchmark) — i.e. `LTV(h) / 3`.

In [ ]:

HORIZONS = [6, 12, 24]
TARGET_RATIO = 3

def ltv_at(channel, month):
    curve = extrap[channel]["curve"]
    return curve.get(month, np.nan)

cac_table = []
for channel in cohort_size.index:
    row = {"channel": channel, "users_acquired": int(cohort_size[channel])}
    for h in HORIZONS:
        ltv_h = ltv_at(channel, h)
        row[f"LTV_{h}mo_NTD"] = round(ltv_h)
        row[f"breakeven_CAC_{h}mo_NTD"] = round(ltv_h)
        row[f"target_max_CAC_{h}mo_3to1_NTD"] = round(ltv_h / TARGET_RATIO)
    cac_table.append(row)

cac_df = pd.DataFrame(cac_table).set_index("channel")
cac_df


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)
for ax, h in zip(axes, HORIZONS):
    channels = cohort_size.index.tolist()
    breakeven = [cac_df.loc[c, f"breakeven_CAC_{h}mo_NTD"] for c in channels]
    target = [cac_df.loc[c, f"target_max_CAC_{h}mo_3to1_NTD"] for c in channels]
    y = np.arange(len(channels))
    ax.barh(y + 0.2, breakeven, height=0.35, color="#93c5fd", label="Break-even CAC")
    ax.barh(y - 0.2, target, height=0.35, color="#2563eb", label="Target CAC (3:1 LTV:CAC)")
    ax.set_yticks(y)
    ax.set_yticklabels(channels)
    ax.set_title(f"{h}-month horizon")
    ax.set_xlabel("NTD")
    if h == HORIZONS[0]:
        ax.legend(loc="lower right", fontsize=8, frameon=False)
plt.suptitle("Max Affordable CAC by Channel and Payback Horizon", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()


## 8. Summary

**How to read this:**
- **Break-even CAC** is the ceiling — spend above this and you lose money on that channel at that horizon, full stop.
- **Target CAC (3:1)** is a healthier bar most subscription businesses actually budget to, leaving margin for infrastructure, support, and profit.
- Channels differ substantially in retention quality (see the retention chart), which is the entire reason channel-level CAC discipline matters — a single blended CAC target across all channels will systematically overpay for low-retention channels and underinvest in high-retention ones.

**Caveats of this analysis:**
- Synthetic data — directionally realistic but not real user behavior; rerun on the actual Kaggle files once accessible for real numbers.
- LTV extrapolation assumes constant exponential decay and steady-state ARPU beyond the reliable window; real retention curves often flatten (a "long-tail" of very sticky survivors), which would make this extrapolation conservative (an underestimate) at long horizons.
- No actual marketing spend data exists in this dataset, so this notebook computes the CAC *ceiling*, not actual channel efficiency (LTV:CAC using real spend) — pair this with real acquisition cost data per channel to get true unit economics.


## 9. Six-month LTV out-of-time validation

The validation script retains only signup cohorts with a full 180-day observation window. Earlier signup-month cohorts are used to estimate a channel-level historical revenue baseline; later matured cohorts are the chronological hold-out. This is not a random subscriber split and does not claim to validate the 24-month decay extrapolation.

The output reports MAE, RMSE, WAPE and mean forecast bias. Contribution-margin sensitivity is displayed separately because margin is not supplied in the synthetic inputs.


In [ ]:
%run scripts/validate_ltv.py

import json
from pathlib import Path
import pandas as pd
oot = json.loads(Path('results/ltv/out_of_time_validation.json').read_text())
display(pd.Series(oot).to_frame('value'))
display(pd.read_csv('results/ltv/out_of_time_by_cohort.csv'))
display(pd.read_csv('results/ltv/contribution_margin_sensitivity.csv'))
